# Preprocessing Final — Detector, Crop, YOLO26 Pose, dan Frame-XY

```text
video mentah
→ sampling sekitar 30 FPS
→ YOLO person detector
→ crop internal dengan padding
→ YOLO26s Pose pada crop
→ mapping keypoint ke koordinat frame asli
→ normalisasi frame_xy (x/W, y/H)
→ interpolasi gap internal maksimal 15 frame
→ active-region crop
→ window 30, step 10
→ split berdasarkan video sumber
→ dataset akhir (N, 30, 51)
```

Bounding box detector asli dipakai untuk memilih target. Bounding box dengan padding hanya dipakai untuk crop internal dan tidak menjadi fitur model.


## 1. Install library

In [1]:
# Jangan upgrade Pandas/OpenCV/Scikit-learn bawaan Kaggle.
%pip install -q ultralytics lap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 55.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## 2. Import

In [2]:
from __future__ import annotations

import hashlib
import json
import math
import re
import shutil
import subprocess
import time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from ultralytics import YOLO

print("OpenCV :", cv2.__version__)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
OpenCV : 4.13.0
PyTorch: 2.10.0+cu128
CUDA   : True


## 3. Konfigurasi

In [3]:

# ============================================================
# KONFIGURASI FINAL
# ============================================================

KAGGLE_INPUT_ROOT = Path("/kaggle/input")

RAW_DATASET_ROOT = Path(
    "/kaggle/input/datasets/wafabila/ucf-sendiri/Dataset taking sendiri"
)

OUTPUT_DIR = Path(
    "/kaggle/working/preprocessed_final_detector_crop_yolopose_framexy_30x51"
)

ARRAY_DIR = OUTPUT_DIR / "arrays"
REPORT_DIR = OUTPUT_DIR / "reports"
CACHE_DIR = OUTPUT_DIR / "video_pose_cache"
PREVIEW_DIR = OUTPUT_DIR / "preview"
TRANSCODE_DIR = OUTPUT_DIR / "transcoded_fallback"

for folder in [
    OUTPUT_DIR,
    ARRAY_DIR,
    REPORT_DIR,
    CACHE_DIR,
    PREVIEW_DIR,
    TRANSCODE_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

ACTIVITY_CLASSES = [
    "boxing",
    "carrying",
    "clapping",
    "digging",
    "jogging",
    "running",
    "throwing",
    "walking",
    "waving",
]

CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(ACTIVITY_CLASSES)
}

VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv",
    ".mpeg", ".mpg", ".m4v",
}

# Model
DETECTOR_MODEL_NAME = "yolov8s.pt"
POSE_MODEL_NAME = "yolo26s-pose.pt"
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# Detector manusia
DETECTOR_CONF = 0.15
DETECTOR_IMGSZ = 960
DETECTOR_MAX_DET = 10

# Crop internal
CROP_PAD_X = 0.25
CROP_PAD_Y = 0.35

# Pose pada crop
POSE_CONF = 0.05
POSE_IOU = 0.50
POSE_IMGSZ_CROP = 640
POSE_MAX_DET_CROP = 5

# Sampling dan sequence
TARGET_FPS = 30.0
WINDOW_SIZE = 30
STEP_SIZE = 10

# Fitur
NUM_KEYPOINTS = 17
RAW_FEATURE_DIM = 51

# Validitas pose
KEYPOINT_CONF_THRESHOLD = 0.15
MIN_VALID_KEYPOINTS = 5
MIN_VALID_FRAMES_PER_VIDEO = 1
MIN_VALID_RATIO_PER_WINDOW = 0.20
MIN_VALID_FRAMES_PER_WINDOW = max(
    1,
    int(math.ceil(WINDOW_SIZE * MIN_VALID_RATIO_PER_WINDOW)),
)

MAX_INTERPOLATION_GAP = 15
ACTIVE_MARGIN_FRAMES = 15

# Split berbasis video
SPLIT_SEED = 42
TARGET_SPLIT_COUNTS = {
    "train": 749,
    "validation": 147,
    "test": 100,
}

DEBUG_MODE = False
DEBUG_MAX_VIDEO_PER_CLASS = 2

PIPELINE_VERSION = "detector_crop_yolo26pose_framexy_v1"

print("Output      :", OUTPUT_DIR)
print("Pipeline    :", PIPELINE_VERSION)
print("Padding     :", CROP_PAD_X, CROP_PAD_Y)
print("Window/step :", WINDOW_SIZE, STEP_SIZE)


Output      : /kaggle/working/preprocessed_final_detector_crop_yolopose_framexy_30x51
Pipeline    : detector_crop_yolo26pose_framexy_v1
Padding     : 0.25 0.35
Window/step : 30 10


## 4. Cari folder video mentah secara otomatis

In [4]:
def normalize_name(
    text,
):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(text).lower(),
    )


def contains_all_class_folders(
    root: Path,
) -> bool:
    if not root.is_dir():
        return False

    child_names = {
        normalize_name(path.name)
        for path in root.iterdir()
        if path.is_dir()
    }

    return all(
        normalize_name(class_name)
        in child_names
        for class_name
        in ACTIVITY_CLASSES
    )


def find_raw_dataset_root() -> Path:
    if (
        RAW_DATASET_ROOT.exists()
        and contains_all_class_folders(
            RAW_DATASET_ROOT
        )
    ):
        return RAW_DATASET_ROOT

    candidates = []

    for directory in KAGGLE_INPUT_ROOT.rglob("*"):
        if (
            directory.is_dir()
            and contains_all_class_folders(
                directory
            )
        ):
            candidates.append(directory)

    if not candidates:
        raise FileNotFoundError(
            "Folder video mentah dengan 9 kelas "
            "tidak ditemukan di /kaggle/input."
        )

    candidates.sort(
        key=lambda path: (
            len(path.parts),
            str(path),
        )
    )

    return candidates[0]


DATASET_ROOT = find_raw_dataset_root()

print("Dataset video mentah ditemukan:")
print(DATASET_ROOT)

Dataset video mentah ditemukan:
/kaggle/input/datasets/wafabila/ucf-sendiri/Dataset taking sendiri


## 5. Buat manifest seluruh video

In [5]:
def find_class_dir(
    class_name: str,
) -> Path:
    direct = (
        DATASET_ROOT
        / class_name
    )

    if direct.exists():
        return direct

    target = normalize_name(
        class_name
    )

    matches = [
        path
        for path in DATASET_ROOT.iterdir()
        if (
            path.is_dir()
            and normalize_name(
                path.name
            ) == target
        )
    ]

    if not matches:
        raise FileNotFoundError(
            f"Folder kelas tidak ditemukan: "
            f"{class_name}"
        )

    return matches[0]


def find_videos(
    class_dir: Path,
):
    return sorted(
        path
        for path in class_dir.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in VIDEO_EXTENSIONS
        )
    )


manifest_rows = []

for class_name in ACTIVITY_CLASSES:
    class_dir = find_class_dir(
        class_name
    )

    video_paths = find_videos(
        class_dir
    )

    if DEBUG_MODE:
        video_paths = video_paths[
            :DEBUG_MAX_VIDEO_PER_CLASS
        ]

    for video_path in video_paths:
        source_video_id = (
            f"{class_name}/"
            f"{video_path.stem}"
        )

        manifest_rows.append({
            "class_name": class_name,
            "label": (
                CLASS_TO_INDEX[
                    class_name
                ]
            ),
            "source_video_id": (
                source_video_id
            ),
            "filename": (
                video_path.name
            ),
            "video_path": (
                str(video_path)
            ),
        })

video_manifest_df = pd.DataFrame(
    manifest_rows
)

if video_manifest_df.empty:
    raise RuntimeError(
        "Tidak ada video ditemukan."
    )

duplicate_ids = (
    video_manifest_df[
        "source_video_id"
    ].duplicated()
)

if duplicate_ids.any():
    duplicates = (
        video_manifest_df.loc[
            duplicate_ids,
            "source_video_id",
        ].tolist()
    )

    raise RuntimeError(
        f"source_video_id duplikat: "
        f"{duplicates[:20]}"
    )

print(
    "Total video:",
    len(video_manifest_df),
)

display(
    video_manifest_df[
        "class_name"
    ].value_counts()
    .sort_index()
    .rename("video_count")
    .to_frame()
)

video_manifest_df.to_csv(
    REPORT_DIR
    / "video_manifest_input.csv",
    index=False,
)

Total video: 996


,video_count
class_name,
boxing,110
carrying,113
clapping,109
digging,109
jogging,109
running,121
throwing,108
walking,109
waving,108


## 6. Buat split train, validation, dan test berdasarkan video

In [6]:
def allocate_class_counts(
    class_sizes: dict[str, int],
    target_total: int,
) -> dict[str, int]:
    total = sum(
        class_sizes.values()
    )

    raw = {
        class_name: (
            size
            * target_total
            / total
        )
        for class_name, size
        in class_sizes.items()
    }

    allocated = {
        class_name: int(
            math.floor(value)
        )
        for class_name, value
        in raw.items()
    }

    remaining = (
        target_total
        - sum(
            allocated.values()
        )
    )

    order = sorted(
        raw,
        key=lambda name: (
            raw[name]
            - allocated[name]
        ),
        reverse=True,
    )

    for class_name in order[
        :remaining
    ]:
        allocated[
            class_name
        ] += 1

    return allocated


total_videos = len(
    video_manifest_df
)

if (
    not DEBUG_MODE
    and total_videos
    != sum(
        TARGET_SPLIT_COUNTS.values()
    )
):
    raise RuntimeError(
        f"Jumlah video ditemukan {total_videos}, "
        f"sedangkan pipeline final mengharapkan "
        f"{sum(TARGET_SPLIT_COUNTS.values())} video."
    )

class_sizes = (
    video_manifest_df[
        "class_name"
    ].value_counts()
    .to_dict()
)

test_alloc = allocate_class_counts(
    class_sizes,
    (
        TARGET_SPLIT_COUNTS["test"]
        if not DEBUG_MODE
        else max(
            1,
            round(
                total_videos * 0.10
            ),
        )
    ),
)

remaining_after_test = {
    class_name: (
        class_sizes[class_name]
        - test_alloc[class_name]
    )
    for class_name
    in ACTIVITY_CLASSES
}

validation_alloc = allocate_class_counts(
    remaining_after_test,
    (
        TARGET_SPLIT_COUNTS[
            "validation"
        ]
        if not DEBUG_MODE
        else max(
            1,
            round(
                total_videos * 0.15
            ),
        )
    ),
)

rng = np.random.default_rng(
    SPLIT_SEED
)

split_by_video = {}

for class_name in ACTIVITY_CLASSES:
    class_rows = (
        video_manifest_df[
            video_manifest_df[
                "class_name"
            ] == class_name
        ]
        .sort_values(
            "source_video_id"
        )
        .copy()
    )

    indices = class_rows.index.to_numpy(
        copy=True
    )

    rng.shuffle(indices)

    n_test = test_alloc[
        class_name
    ]

    n_validation = validation_alloc[
        class_name
    ]

    test_indices_class = indices[
        :n_test
    ]

    validation_indices_class = indices[
        n_test:
        n_test + n_validation
    ]

    train_indices_class = indices[
        n_test + n_validation:
    ]

    for index in test_indices_class:
        split_by_video[
            video_manifest_df.loc[
                index,
                "source_video_id",
            ]
        ] = "test"

    for index in validation_indices_class:
        split_by_video[
            video_manifest_df.loc[
                index,
                "source_video_id",
            ]
        ] = "validation"

    for index in train_indices_class:
        split_by_video[
            video_manifest_df.loc[
                index,
                "source_video_id",
            ]
        ] = "train"

video_manifest_df["split"] = (
    video_manifest_df[
        "source_video_id"
    ].map(
        split_by_video
    )
)

if (
    video_manifest_df[
        "split"
    ].isna().any()
):
    raise RuntimeError(
        "Ada video yang belum memperoleh split."
    )

split_counts = (
    video_manifest_df[
        "split"
    ].value_counts()
    .to_dict()
)

print("Split video:", split_counts)

if (
    not DEBUG_MODE
    and split_counts
    != TARGET_SPLIT_COUNTS
):
    raise RuntimeError(
        f"Split bukan 749/147/100: "
        f"{split_counts}"
    )

video_manifest_df.to_csv(
    REPORT_DIR
    / "video_split_manifest.csv",
    index=False,
)

display(
    pd.crosstab(
        video_manifest_df[
            "class_name"
        ],
        video_manifest_df[
            "split"
        ],
    )
)

Split video: {'train': 749, 'validation': 147, 'test': 100}


split,test,train,validation
class_name,,,
boxing,11,83,16
carrying,11,85,17
clapping,11,82,16
digging,11,82,16
jogging,11,82,16
running,12,91,18
throwing,11,81,16
walking,11,82,16
waving,11,81,16


## 7. Load YOLO Pose

In [7]:

detector_model = YOLO(DETECTOR_MODEL_NAME)
pose_model = YOLO(POSE_MODEL_NAME)

print("Detector siap:", DETECTOR_MODEL_NAME)
print("Pose siap    :", POSE_MODEL_NAME)
print("Device       :", DEVICE)


Detector siap: yolov8s.pt
Pose siap    : yolo26s-pose.pt
Device       : 0


## 8. Fungsi pembacaan video dan sampling frame

In [8]:
def make_safe_filename(
    source_video_id: str,
) -> str:
    digest = hashlib.sha1(
        source_video_id.encode(
            "utf-8"
        )
    ).hexdigest()[:10]

    base = re.sub(
        r"[^a-zA-Z0-9_-]+",
        "_",
        source_video_id,
    ).strip("_")

    return (
        f"{base}_{digest}.npz"
    )


def transcode_video(
    video_path: Path,
) -> Path | None:
    output_path = (
        TRANSCODE_DIR
        / (
            make_safe_filename(
                str(video_path)
            ).replace(
                ".npz",
                ".mp4",
            )
        )
    )

    command = [
        "ffmpeg",
        "-y",
        "-loglevel",
        "error",
        "-i",
        str(video_path),
        "-c:v",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        "-an",
        str(output_path),
    ]

    try:
        subprocess.run(
            command,
            check=True,
        )

        return (
            output_path
            if output_path.exists()
            else None
        )

    except Exception:
        return None


def open_video(
    video_path: Path,
):
    capture = cv2.VideoCapture(
        str(video_path)
    )

    if capture.isOpened():
        return capture, video_path, False

    capture.release()

    transcoded_path = transcode_video(
        video_path
    )

    if transcoded_path is None:
        return None, video_path, False

    capture = cv2.VideoCapture(
        str(transcoded_path)
    )

    if not capture.isOpened():
        capture.release()
        return None, video_path, True

    return capture, transcoded_path, True


def calculate_sampling_stride(
    source_fps: float,
) -> float:
    if (
        not np.isfinite(
            source_fps
        )
        or source_fps <= 0
    ):
        source_fps = TARGET_FPS

    return max(
        source_fps / TARGET_FPS,
        1.0,
    )

## 9. Fungsi pemilihan orang dan normalisasi pose

In [9]:

def box_area(box):
    if box is None:
        return 0.0
    x1, y1, x2, y2 = box
    return max(0.0, float(x2 - x1)) * max(0.0, float(y2 - y1))


def box_iou(box_a, box_b):
    if box_a is None or box_b is None:
        return 0.0

    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    union = box_area(box_a) + box_area(box_b) - intersection

    return intersection / union if union > 1e-6 else 0.0


def select_person_candidate(boxes, confidences, previous_box):
    if len(boxes) == 0:
        return None, 0.0

    if previous_box is None:
        areas = np.asarray([box_area(box) for box in boxes], dtype=np.float32)
        scores = areas * np.maximum(confidences, 1e-6)
    else:
        scores = np.asarray([
            4.0 * box_iou(previous_box, box)
            + 0.01 * math.log1p(box_area(box))
            + 0.25 * float(confidence)
            for box, confidence in zip(boxes, confidences)
        ], dtype=np.float32)

    index = int(np.argmax(scores))
    return boxes[index].astype(np.float32), float(confidences[index])


def detect_main_person(frame, previous_box=None):
    result = detector_model.predict(
        source=frame,
        imgsz=DETECTOR_IMGSZ,
        conf=DETECTOR_CONF,
        classes=[0],
        max_det=DETECTOR_MAX_DET,
        device=DEVICE,
        verbose=False,
    )[0]

    if result.boxes is None or len(result.boxes) == 0:
        return None, 0.0

    boxes = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
    confidences = result.boxes.conf.detach().cpu().numpy().astype(np.float32)

    return select_person_candidate(boxes, confidences, previous_box)


def crop_with_padding(frame, person_box):
    if person_box is None:
        return None, None

    frame_height, frame_width = frame.shape[:2]
    x1, y1, x2, y2 = map(float, person_box)
    box_width = x2 - x1
    box_height = y2 - y1

    crop_x1 = max(0, int(math.floor(x1 - box_width * CROP_PAD_X)))
    crop_y1 = max(0, int(math.floor(y1 - box_height * CROP_PAD_Y)))
    crop_x2 = min(frame_width, int(math.ceil(x2 + box_width * CROP_PAD_X)))
    crop_y2 = min(frame_height, int(math.ceil(y2 + box_height * CROP_PAD_Y)))

    if crop_x2 <= crop_x1 or crop_y2 <= crop_y1:
        return None, None

    crop = frame[crop_y1:crop_y2, crop_x1:crop_x2].copy()
    crop_box = np.asarray(
        [crop_x1, crop_y1, crop_x2, crop_y2],
        dtype=np.float32,
    )

    return crop, crop_box


def run_pose_on_crop(crop):
    if crop is None or crop.size == 0:
        return None, 0.0

    result = pose_model.predict(
        source=crop,
        imgsz=POSE_IMGSZ_CROP,
        conf=POSE_CONF,
        iou=POSE_IOU,
        max_det=POSE_MAX_DET_CROP,
        classes=[0],
        device=DEVICE,
        verbose=False,
    )[0]

    if (
        result.boxes is None
        or len(result.boxes) == 0
        or result.keypoints is None
        or len(result.keypoints.data) == 0
    ):
        return None, 0.0

    boxes = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
    confidences = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
    keypoints = result.keypoints.data.detach().cpu().numpy().astype(np.float32)

    areas = (
        (boxes[:, 2] - boxes[:, 0])
        * (boxes[:, 3] - boxes[:, 1])
    )
    index = int(np.argmax(areas * np.maximum(confidences, 1e-6)))

    return keypoints[index], float(confidences[index])


def map_crop_keypoints_to_frame(crop_keypoints, crop_box):
    if crop_keypoints is None or crop_box is None:
        return None

    keypoints = np.asarray(crop_keypoints, dtype=np.float32).copy()

    if keypoints.shape != (NUM_KEYPOINTS, 3):
        return None

    keypoints[:, 0] += float(crop_box[0])
    keypoints[:, 1] += float(crop_box[1])

    return keypoints


def normalize_pose_frame_xy(frame_keypoints, frame_shape):
    output = np.zeros((NUM_KEYPOINTS, 3), dtype=np.float32)

    if frame_keypoints is None:
        return output, False

    keypoints = np.asarray(frame_keypoints, dtype=np.float32)

    if keypoints.shape != (NUM_KEYPOINTS, 3):
        return output, False

    if not np.isfinite(keypoints).all():
        return output, False

    valid = keypoints[:, 2] >= KEYPOINT_CONF_THRESHOLD

    if int(valid.sum()) < MIN_VALID_KEYPOINTS:
        return output, False

    frame_height, frame_width = frame_shape[:2]

    if frame_width <= 0 or frame_height <= 0:
        return output, False

    output[:, 0] = keypoints[:, 0] / float(frame_width)
    output[:, 1] = keypoints[:, 1] / float(frame_height)
    output[:, 2] = np.clip(keypoints[:, 2], 0.0, 1.0)
    output[~valid, :] = 0.0

    return output, True


## 10. Interpolasi gap pendek dan crop area aktif

In [10]:
def interpolate_short_internal_gaps(
    features,
    valid_mask,
):
    features = features.copy()
    valid_mask = valid_mask.copy()

    valid_indices = np.where(
        valid_mask
    )[0]

    if len(valid_indices) < 2:
        return (
            features,
            valid_mask,
        )

    for left_index, right_index in zip(
        valid_indices[:-1],
        valid_indices[1:],
    ):
        gap = (
            right_index
            - left_index
            - 1
        )

        if (
            gap <= 0
            or gap
            > MAX_INTERPOLATION_GAP
        ):
            continue

        left_pose = features[
            left_index
        ]

        right_pose = features[
            right_index
        ]

        for offset in range(
            1,
            gap + 1,
        ):
            alpha = (
                offset
                / (gap + 1)
            )

            interpolated = (
                (1.0 - alpha)
                * left_pose
                + alpha
                * right_pose
            )

            # Confidence hasil interpolasi tidak boleh
            # melebihi confidence endpoint terlemah.
            interpolated[:, 2] = np.minimum(
                left_pose[:, 2],
                right_pose[:, 2],
            )

            features[
                left_index + offset
            ] = interpolated

            valid_mask[
                left_index + offset
            ] = True

    return (
        features,
        valid_mask,
    )


def crop_to_active_region(
    features,
    valid_mask,
):
    valid_indices = np.where(
        valid_mask
    )[0]

    if len(valid_indices) == 0:
        return (
            features,
            valid_mask,
            0,
        )

    start = max(
        0,
        int(valid_indices[0])
        - ACTIVE_MARGIN_FRAMES,
    )

    end = min(
        len(features),
        int(valid_indices[-1])
        + ACTIVE_MARGIN_FRAMES
        + 1,
    )

    return (
        features[start:end],
        valid_mask[start:end],
        start,
    )

## Validasi Pipeline Baru

In [11]:

assert NUM_KEYPOINTS == 17
assert RAW_FEATURE_DIM == 51
assert WINDOW_SIZE == 30
assert STEP_SIZE == 10
assert CROP_PAD_X == 0.25
assert CROP_PAD_Y == 0.35

print("Pipeline valid:")
print("YOLO detector → crop → YOLO26 Pose → mapping → frame_xy")


Pipeline valid:
YOLO detector → crop → YOLO26 Pose → mapping → frame_xy


## 11. Ekstraksi pose dari satu video

In [12]:

def extract_pose_from_video(video_path: Path):
    capture, actual_path, used_transcode = open_video(video_path)

    empty_features = np.zeros((0, NUM_KEYPOINTS, 3), dtype=np.float32)
    empty_mask = np.zeros((0,), dtype=bool)

    if capture is None:
        return {
            "status": "VIDEO_OPEN_FAILED",
            "features": empty_features,
            "valid_mask": empty_mask,
            "source_fps": 0.0,
            "sampled_frames": 0,
            "valid_frames": 0,
            "used_transcode": used_transcode,
        }

    source_fps = float(capture.get(cv2.CAP_PROP_FPS))
    sampling_stride = calculate_sampling_stride(source_fps)

    features = []
    valid_mask = []

    source_frame_index = 0
    next_sample_position = 0.0
    previous_person_box = None

    while True:
        ok, frame = capture.read()

        if not ok:
            break

        should_sample = source_frame_index + 1e-9 >= next_sample_position

        if should_sample:
            normalized_pose = np.zeros(
                (NUM_KEYPOINTS, 3),
                dtype=np.float32,
            )
            pose_valid = False

            person_box, detector_confidence = detect_main_person(
                frame,
                previous_box=previous_person_box,
            )

            if person_box is not None:
                crop, crop_box = crop_with_padding(frame, person_box)
                crop_keypoints, pose_confidence = run_pose_on_crop(crop)

                frame_keypoints = map_crop_keypoints_to_frame(
                    crop_keypoints,
                    crop_box,
                )

                normalized_pose, pose_valid = normalize_pose_frame_xy(
                    frame_keypoints,
                    frame.shape,
                )

                if pose_valid:
                    previous_person_box = person_box.copy()

            features.append(normalized_pose)
            valid_mask.append(bool(pose_valid))
            next_sample_position += sampling_stride

        source_frame_index += 1

    capture.release()

    features = np.asarray(features, dtype=np.float32)
    valid_mask = np.asarray(valid_mask, dtype=bool)

    if len(features) == 0:
        return {
            "status": "NO_SAMPLED_FRAME",
            "features": empty_features,
            "valid_mask": empty_mask,
            "source_fps": source_fps,
            "sampled_frames": 0,
            "valid_frames": 0,
            "used_transcode": used_transcode,
        }

    features, valid_mask = interpolate_short_internal_gaps(
        features,
        valid_mask,
    )

    features, valid_mask, active_start = crop_to_active_region(
        features,
        valid_mask,
    )

    valid_frames = int(valid_mask.sum())

    status = (
        "OK"
        if valid_frames >= MIN_VALID_FRAMES_PER_VIDEO
        else "NO_SKELETON"
    )

    return {
        "status": status,
        "features": features,
        "valid_mask": valid_mask,
        "source_fps": source_fps,
        "sampled_frames": int(len(features)),
        "valid_frames": valid_frames,
        "active_start": int(active_start),
        "used_transcode": used_transcode,
        "actual_path": str(actual_path),
    }


## 12. Proses seluruh video dan simpan cache

In [13]:
video_report_rows = []
cache_manifest_rows = []

started = time.time()

records = video_manifest_df.to_dict(
    orient="records"
)

for row in tqdm(
    records,
    total=len(records),
    desc="Ekstraksi YOLO Pose",
):
    video_path = Path(
        row["video_path"]
    )

    source_video_id = str(
        row["source_video_id"]
    )

    cache_filename = make_safe_filename(
        PIPELINE_VERSION + '__' + source_video_id
    )

    cache_path = (
        CACHE_DIR
        / cache_filename
    )

    if cache_path.exists():
        with np.load(
            cache_path,
            allow_pickle=False,
        ) as cached:
            features = cached[
                "features"
            ].astype(np.float32)

            valid_mask = cached[
                "valid_mask"
            ].astype(bool)

        extraction = {
            "status": (
                "OK"
                if int(
                    valid_mask.sum()
                ) >= MIN_VALID_FRAMES_PER_VIDEO
                else "NO_SKELETON"
            ),
            "features": features,
            "valid_mask": valid_mask,
            "source_fps": np.nan,
            "sampled_frames": int(
                len(features)
            ),
            "valid_frames": int(
                valid_mask.sum()
            ),
            "active_start": 0,
            "used_transcode": False,
            "actual_path": str(
                video_path
            ),
        }
    else:
        extraction = extract_pose_from_video(
            video_path
        )

        np.savez_compressed(
            cache_path,
            features=extraction[
                "features"
            ].astype(np.float32),
            valid_mask=extraction[
                "valid_mask"
            ].astype(bool),
        )

    sampled_frames = int(
        extraction[
            "sampled_frames"
        ]
    )

    valid_frames = int(
        extraction[
            "valid_frames"
        ]
    )

    video_report_rows.append({
        "source_video_id": (
            source_video_id
        ),
        "class_name": (
            row["class_name"]
        ),
        "label": int(
            row["label"]
        ),
        "split": row["split"],
        "filename": row["filename"],
        "video_path": row["video_path"],
        "status": extraction[
            "status"
        ],
        "source_fps": extraction.get(
            "source_fps",
            np.nan,
        ),
        "sampled_frames": (
            sampled_frames
        ),
        "valid_frames": (
            valid_frames
        ),
        "valid_ratio": float(
            valid_frames
            / max(
                sampled_frames,
                1,
            )
        ),
        "used_transcode": bool(
            extraction.get(
                "used_transcode",
                False,
            )
        ),
        "cache_file": (
            cache_filename
        ),
    })

    cache_manifest_rows.append({
        "source_video_id": (
            source_video_id
        ),
        "cache_file": (
            cache_filename
        ),
        "cache_path": str(
            cache_path
        ),
    })

video_report_df = pd.DataFrame(
    video_report_rows
)

cache_manifest_df = pd.DataFrame(
    cache_manifest_rows
)

video_report_df.to_csv(
    REPORT_DIR
    / "video_pose_report.csv",
    index=False,
)

cache_manifest_df.to_csv(
    REPORT_DIR
    / "cache_manifest.csv",
    index=False,
)

print(
    "Waktu ekstraksi:",
    f"{(time.time() - started) / 60:.2f} menit",
)

display(
    video_report_df[
        "status"
    ].value_counts()
    .rename("video_count")
    .to_frame()
)


Ekstraksi YOLO Pose:   0%|          | 0/996 [00:00<?, ?it/s]

Waktu ekstraksi: 161.81 menit


,video_count
status,
OK,996


## 13. Bentuk sequence window 30

In [14]:
X_sequences = []
valid_masks = []
y_labels = []
source_video_ids = []
sequence_ids = []
split_values = []
start_frames = []
end_frames = []
valid_frame_counts = []
missing_ratios = []

sequence_rows = []
per_video_sequence_rows = []

manifest_lookup = (
    video_manifest_df.set_index(
        "source_video_id"
    )
)

cache_records = cache_manifest_df.to_dict(
    orient="records"
)

for row in tqdm(
    cache_records,
    total=len(cache_records),
    desc="Membentuk sequence 30",
):
    source_video_id = str(
        row["source_video_id"]
    )

    video_info = manifest_lookup.loc[
        source_video_id
    ]

    with np.load(
        row["cache_path"],
        allow_pickle=False,
    ) as cached:
        features = cached[
            "features"
        ].astype(np.float32)

        valid_mask = cached[
            "valid_mask"
        ].astype(bool)

    if (
        features.ndim != 3
        or features.shape[1:] != (
            NUM_KEYPOINTS,
            3,
        )
    ):
        raise RuntimeError(
            f"Shape cache salah: "
            f"{row['cache_path']} "
            f"{features.shape}"
        )

    flattened = features.reshape(
        len(features),
        RAW_FEATURE_DIM,
    )

    generated = 0

    if len(features) > 0:
        if len(features) <= WINDOW_SIZE:
            starts = [0]
        else:
            starts = list(
                range(
                    0,
                    len(features)
                    - WINDOW_SIZE
                    + 1,
                    STEP_SIZE,
                )
            )

            last_start = (
                len(features)
                - WINDOW_SIZE
            )

            if (
                starts
                and starts[-1]
                != last_start
            ):
                starts.append(
                    last_start
                )

        for start in starts:
            end = (
                start
                + WINDOW_SIZE
            )

            sequence = np.zeros(
                (
                    WINDOW_SIZE,
                    RAW_FEATURE_DIM,
                ),
                dtype=np.float32,
            )

            sequence_mask = np.zeros(
                (
                    WINDOW_SIZE,
                ),
                dtype=bool,
            )

            available_end = min(
                end,
                len(features),
            )

            available_length = max(
                0,
                available_end
                - start,
            )

            if available_length > 0:
                sequence[
                    :available_length
                ] = flattened[
                    start:available_end
                ]

                sequence_mask[
                    :available_length
                ] = valid_mask[
                    start:available_end
                ]

            valid_count = int(
                sequence_mask.sum()
            )

            if (
                valid_count
                < MIN_VALID_FRAMES_PER_WINDOW
            ):
                continue

            sequence[
                ~sequence_mask
            ] = 0.0

            missing_ratio = float(
                1.0
                - valid_count
                / WINDOW_SIZE
            )

            sequence_id = (
                re.sub(
                    r"[^a-zA-Z0-9_-]+",
                    "_",
                    source_video_id,
                )
                + f"__w30__s{start:06d}"
            )

            X_sequences.append(
                sequence
            )
            valid_masks.append(
                sequence_mask
            )
            y_labels.append(
                int(
                    video_info[
                        "label"
                    ]
                )
            )
            source_video_ids.append(
                source_video_id
            )
            sequence_ids.append(
                sequence_id
            )
            split_values.append(
                str(
                    video_info[
                        "split"
                    ]
                )
            )
            start_frames.append(
                int(start)
            )
            end_frames.append(
                int(end)
            )
            valid_frame_counts.append(
                valid_count
            )
            missing_ratios.append(
                missing_ratio
            )

            sequence_rows.append({
                "sequence_id": (
                    sequence_id
                ),
                "source_video_id": (
                    source_video_id
                ),
                "class_name": (
                    video_info[
                        "class_name"
                    ]
                ),
                "label": int(
                    video_info[
                        "label"
                    ]
                ),
                "split": (
                    video_info[
                        "split"
                    ]
                ),
                "start_frame": int(
                    start
                ),
                "end_frame": int(
                    end
                ),
                "valid_frames": (
                    valid_count
                ),
                "missing_ratio": (
                    missing_ratio
                ),
            })

            generated += 1

    per_video_sequence_rows.append({
        "source_video_id": (
            source_video_id
        ),
        "class_name": (
            video_info[
                "class_name"
            ]
        ),
        "split": (
            video_info[
                "split"
            ]
        ),
        "sampled_frames": int(
            len(features)
        ),
        "generated_sequences": (
            generated
        ),
        "represented": bool(
            generated > 0
        ),
    })

if not X_sequences:
    raise RuntimeError(
        "Tidak ada sequence yang berhasil dibuat."
    )

X_sequences = np.stack(
    X_sequences
).astype(np.float32)

valid_masks = np.stack(
    valid_masks
).astype(bool)

y_labels = np.asarray(
    y_labels,
    dtype=np.int64,
)

source_video_ids = np.asarray(
    source_video_ids,
    dtype=str,
)

sequence_ids = np.asarray(
    sequence_ids,
    dtype=str,
)

split_array = np.asarray(
    split_values,
    dtype=str,
)

start_frames = np.asarray(
    start_frames,
    dtype=np.int64,
)

end_frames = np.asarray(
    end_frames,
    dtype=np.int64,
)

valid_frame_counts = np.asarray(
    valid_frame_counts,
    dtype=np.int64,
)

missing_ratios = np.asarray(
    missing_ratios,
    dtype=np.float32,
)

print(
    "X shape   :",
    X_sequences.shape,
)

print(
    "Mask shape:",
    valid_masks.shape,
)


Membentuk sequence 30:   0%|          | 0/996 [00:00<?, ?it/s]

X shape   : (27442, 30, 51)
Mask shape: (27442, 30)


## 14. Audit split dan source-video leakage

In [15]:
train_indices = np.where(
    split_array == "train"
)[0].astype(np.int64)

validation_indices = np.where(
    split_array == "validation"
)[0].astype(np.int64)

test_indices = np.where(
    split_array == "test"
)[0].astype(np.int64)

train_videos = set(
    source_video_ids[
        train_indices
    ]
)

validation_videos = set(
    source_video_ids[
        validation_indices
    ]
)

test_videos = set(
    source_video_ids[
        test_indices
    ]
)

overlap = {
    "train_validation": len(
        train_videos
        & validation_videos
    ),
    "train_test": len(
        train_videos
        & test_videos
    ),
    "validation_test": len(
        validation_videos
        & test_videos
    ),
}

represented_video_counts = {
    "train": len(
        train_videos
    ),
    "validation": len(
        validation_videos
    ),
    "test": len(
        test_videos
    ),
}

print(
    "Video terwakili:",
    represented_video_counts,
)

print(
    "Leakage:",
    overlap,
)

if any(
    overlap.values()
):
    raise RuntimeError(
        f"Source-video leakage ditemukan: "
        f"{overlap}"
    )

missing_video_df = pd.DataFrame(
    per_video_sequence_rows
)

missing_video_df[
    ~missing_video_df[
        "represented"
    ]
].to_csv(
    REPORT_DIR
    / "videos_without_sequences.csv",
    index=False,
)

Video terwakili: {'train': 749, 'validation': 147, 'test': 100}
Leakage: {'train_validation': 0, 'train_test': 0, 'validation_test': 0}


## 15. Simpan array dan laporan

In [16]:
arrays = {
    "X_sequences.npy": (
        X_sequences
    ),
    "y_labels.npy": (
        y_labels
    ),
    "valid_masks.npy": (
        valid_masks
    ),
    "source_video_ids.npy": (
        source_video_ids
    ),
    "sequence_ids.npy": (
        sequence_ids
    ),
    "splits.npy": (
        split_array
    ),
    "train_indices.npy": (
        train_indices
    ),
    "validation_indices.npy": (
        validation_indices
    ),
    "test_indices.npy": (
        test_indices
    ),
    "start_frames.npy": (
        start_frames
    ),
    "end_frames.npy": (
        end_frames
    ),
    "valid_frame_counts.npy": (
        valid_frame_counts
    ),
    "missing_ratios.npy": (
        missing_ratios
    ),
}

for filename, array in arrays.items():
    np.save(
        ARRAY_DIR / filename,
        array,
    )

sequence_metadata_df = pd.DataFrame(
    sequence_rows
)

per_video_sequence_df = pd.DataFrame(
    per_video_sequence_rows
)

sequence_metadata_df.to_csv(
    REPORT_DIR
    / "sequence_metadata.csv",
    index=False,
)

per_video_sequence_df.to_csv(
    REPORT_DIR
    / "per_video_sequence_counts.csv",
    index=False,
)

class_mapping = {
    class_name: index
    for class_name, index
    in CLASS_TO_INDEX.items()
}

with (
    OUTPUT_DIR
    / "class_mapping.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        class_mapping,
        file,
        indent=2,
        ensure_ascii=False,
    )

## 16. Ringkasan final

In [17]:
final_summary = {
    "pipeline": (
        "detector_crop_yolopose_framexy_window30"
    ),
    "dataset_root": str(RAW_DATASET_ROOT),
    "pipeline_version": PIPELINE_VERSION,
    "detector_model": DETECTOR_MODEL_NAME,
    "detector_imgsz": DETECTOR_IMGSZ,
    "detector_confidence": DETECTOR_CONF,
    "crop_padding_x": CROP_PAD_X,
    "crop_padding_y": CROP_PAD_Y,
    "pose_model": POSE_MODEL_NAME,
    "pose_imgsz_crop": POSE_IMGSZ_CROP,
    "pose_confidence": POSE_CONF,
    "keypoint_confidence_threshold": KEYPOINT_CONF_THRESHOLD,
    "normalization": "frame_xy",
    "total_input_videos": int(
        len(
            video_manifest_df
        )
    ),
    "used_source_videos": int(
        len(
            np.unique(
                source_video_ids
            )
        )
    ),
    "total_sequences": int(
        len(
            X_sequences
        )
    ),
    "X_shape": list(
        X_sequences.shape
    ),
    "valid_masks_shape": list(
        valid_masks.shape
    ),
    "window_size": (
        WINDOW_SIZE
    ),
    "step_size": (
        STEP_SIZE
    ),
    "target_fps": (
        TARGET_FPS
    ),
    "observation_seconds": (
        WINDOW_SIZE
        / TARGET_FPS
    ),
    "raw_feature_dim": (
        RAW_FEATURE_DIM
    ),
    "minimum_valid_frames_per_window": (
        MIN_VALID_FRAMES_PER_WINDOW
    ),
    "split_counts_sequences": {
        "train": int(
            len(
                train_indices
            )
        ),
        "validation": int(
            len(
                validation_indices
            )
        ),
        "test": int(
            len(
                test_indices
            )
        ),
    },
    "split_counts_videos": (
        represented_video_counts
    ),
    "source_video_leakage": (
        overlap
    ),
    "selection_reason": (
        "Window 30 frame memberi konteks sekitar "
        "satu detik pada 30 FPS, memberikan hasil "
        "video-level terbaik pada eksperimen, serta "
        "lebih responsif untuk implementasi drone."
    ),
}

with (
    OUTPUT_DIR
    / "preprocessing_summary.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(
    json.dumps(
        final_summary,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "pipeline": "detector_crop_yolopose_framexy_window30",
  "dataset_root": "/kaggle/input/datasets/wafabila/ucf-sendiri/Dataset taking sendiri",
  "pipeline_version": "detector_crop_yolo26pose_framexy_v1",
  "detector_model": "yolov8s.pt",
  "detector_imgsz": 960,
  "detector_confidence": 0.15,
  "crop_padding_x": 0.25,
  "crop_padding_y": 0.35,
  "pose_model": "yolo26s-pose.pt",
  "pose_imgsz_crop": 640,
  "pose_confidence": 0.05,
  "keypoint_confidence_threshold": 0.15,
  "normalization": "frame_xy",
  "total_input_videos": 996,
  "used_source_videos": 996,
  "total_sequences": 27442,
  "X_shape": [
    27442,
    30,
    51
  ],
  "valid_masks_shape": [
    27442,
    30
  ],
  "window_size": 30,
  "step_size": 10,
  "target_fps": 30.0,
  "observation_seconds": 1.0,
  "raw_feature_dim": 51,
  "minimum_valid_frames_per_window": 6,
  "split_counts_sequences": {
    "train": 20630,
    "validation": 4034,
    "test": 2778
  },
  "split_counts_videos": {
    "train": 749,
    "validat

## 17. ZIP seluruh output

In [18]:
zip_path = shutil.make_archive(
    "/kaggle/working/"
    "preprocessed_final_detector_crop_yolopose_framexy_30x51",
    "zip",
    root_dir=OUTPUT_DIR,
)

print("ZIP:", zip_path)

ZIP: /kaggle/working/preprocessed_final_detector_crop_yolopose_framexy_30x51.zip
